# Huricaines Modeling Notebook

Explore merged huricaines tracks, run EDA, train/evaluate the PyTorch model, and save an artifact.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parents[2]
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from src.disasters.models.huricaines import FEATURE_NAMES
from src.disasters.models.huricaines import save_model_bundle
from src.disasters.scripts.huricaines.train_model import (
    load_raw_dataset,
    prepare_training_dataframe,
    split_dataset,
    train_model,
)

In [ ]:
raw_path = ROOT / 'src' / 'disasters' / 'data' / 'huricaines' / 'raw' / 'huricaines_tracks_merged.csv'
if not raw_path.exists():
    raise FileNotFoundError(
        f'{raw_path} is missing. Run: python -m src.disasters.scripts.huricaines.download_data'
    )

raw_df = load_raw_dataset(raw_path)
train_df = prepare_training_dataframe(raw_df)
print(f'Rows: {len(train_df)}')
print(raw_df['source'].value_counts())
train_df.head()

In [ ]:
target_rate = train_df['target'].mean()
print(f'RI-positive class rate: {target_rate:.4f}')

train_df[FEATURE_NAMES].describe().T

In [ ]:
axes = train_df[FEATURE_NAMES].hist(figsize=(11, 7), bins=28)
plt.suptitle('Feature Distributions')
plt.tight_layout()

x_train, y_train, x_val, y_val = split_dataset(train_df, seed=42)
model, feature_mean, feature_std, val_accuracy, val_balanced_accuracy, val_auc = train_model(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    epochs=140,
    batch_size=512,
    learning_rate=9e-4,
    weight_decay=1e-4,
    seed=42,
)
print(f'Validation accuracy: {val_accuracy:.4f}')
print(f'Validation balanced accuracy: {val_balanced_accuracy:.4f}')
print(f'Validation AUROC: {val_auc:.4f}')

In [ ]:
model_path = ROOT / 'src' / 'disasters' / 'models' / 'huricaines.pt'
model_path.parent.mkdir(parents=True, exist_ok=True)

save_model_bundle(
    path=model_path,
    model=model,
    feature_mean=feature_mean,
    feature_std=feature_std,
    model_version='0.5.2-notebook',
    val_accuracy=val_accuracy,
    val_balanced_accuracy=val_balanced_accuracy,
    val_auc=val_auc,
    dataset_rows=int(len(train_df)),
)
print(f'Saved model artifact to: {model_path}')